In [1]:
!pip install transformers accelerate peft bitsandbytes datasets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 42.7 MB/s eta 0:00:00


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from datasets import load_dataset

In [3]:
BASE_MODEL = "codellama/CodeLlama-7b-Instruct-hf"
SFT_ADAPTER = "SongTonyLi/codellama-4bit-peft-lora-r32-sft-c2x86"
FEWSHOT_ADAPTER = "AzizZeynalli/codellama-4bit-peft-lora-r32-sft-c2x86-fewshot"
GRPO_ADAPTER = "SongTonyLi/codellama-grpo-clip-0.1-peft-lora-r32-c2x86"
PPO_ADAPTER = "AzizZeynalli/codellama-7b-c2x86-ppo-rl"
DATASET = "AzizZeynalli/c2x86-dataset"

EXAMPLE_INDICES = [160, 340]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

Device: cuda


In [4]:
dataset = load_dataset(DATASET)
val_data = dataset["validation"]

examples = [val_data[i] for i in EXAMPLE_INDICES]
print(f"Selected {len(examples)} examples: {EXAMPLE_INDICES}")
for i, ex in enumerate(examples):
    print(f"\nExample {i+1}: {ex['c'][:80]}...")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/458 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/10.5M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/2.28M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8134 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2034 [00:00<?, ? examples/s]

Selected 2 examples: [160, 340]

Example 1:  



#include <stdio.h>
int main() {
    int arr[] = {1,2,3,4,5};
    int n =5;
...

Example 2: #include <stdio.h>
int main() {
    int t;
    scanf("%d",&t);
    while(t--){
 ...


In [5]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

In [6]:
def generate_assembly(model, tokenizer, c_code, max_new_tokens=512):
    prompt = f"Translate the following C code to x86 assembly:\n{c_code}\nAssembly:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "Assembly:" in generated:
        generated = generated.split("Assembly:")[-1].strip()

    return generated

In [7]:
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)

base_outputs = []
for i, ex in enumerate(examples):
    print(f"Generating base output for example {i+1}...")
    output = generate_assembly(base_model, tokenizer, ex["c"])
    base_outputs.append(output)

del base_model
torch.cuda.empty_cache()

config.json:   0%|          | 0.00/646 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Generating base output for example 1...
Generating base output for example 2...


In [8]:
sft_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
sft_model = PeftModel.from_pretrained(sft_base, SFT_ADAPTER)

sft_outputs = []
for i, ex in enumerate(examples):
    print(f"Generating SFT output for example {i+1}...")
    output = generate_assembly(sft_model, tokenizer, ex["c"])
    sft_outputs.append(output)

del sft_model, sft_base
torch.cuda.empty_cache()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

adapter_config.json:   0%|          | 0.00/943 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/320M [00:00<?, ?B/s]

Generating SFT output for example 1...
Generating SFT output for example 2...


In [9]:
fewshot_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
fewshot_model = PeftModel.from_pretrained(fewshot_base, FEWSHOT_ADAPTER)

fewshot_outputs = []
for i, ex in enumerate(examples):
    print(f"Generating Few-shot output for example {i+1}...")
    output = generate_assembly(fewshot_model, tokenizer, ex["c"])
    fewshot_outputs.append(output)

del fewshot_model, fewshot_base
torch.cuda.empty_cache()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/320M [00:00<?, ?B/s]

Generating Few-shot output for example 1...
Generating Few-shot output for example 2...


In [10]:
grpo_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
grpo_model = PeftModel.from_pretrained(grpo_base, GRPO_ADAPTER)

grpo_outputs = []
for i, ex in enumerate(examples):
    print(f"Generating GRPO output for example {i+1}...")
    output = generate_assembly(grpo_model, tokenizer, ex["c"])
    grpo_outputs.append(output)

del grpo_model, grpo_base
torch.cuda.empty_cache()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/320M [00:00<?, ?B/s]

Generating GRPO output for example 1...
Generating GRPO output for example 2...


In [11]:
ppo_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
ppo_model = PeftModel.from_pretrained(ppo_base, PPO_ADAPTER)

ppo_outputs = []
for i, ex in enumerate(examples):
    print(f"Generating PPO output for example {i+1}...")
    output = generate_assembly(ppo_model, tokenizer, ex["c"])
    ppo_outputs.append(output)

del ppo_model, ppo_base
torch.cuda.empty_cache()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

adapter_config.json:   0%|          | 0.00/734 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/320M [00:00<?, ?B/s]

Generating PPO output for example 1...
Generating PPO output for example 2...


In [12]:
def truncate_output(text, max_lines=30):
    lines = text.strip().split('\n')
    if len(lines) > max_lines:
        return '\n'.join(lines[:max_lines]) + '\n    ...'
    return text

for i, ex in enumerate(examples):
    print(f"\n{'='*80}")
    print(f"EXAMPLE {i+1}: {'GCD' if i==0 else 'Factorial'}")
    print("="*80)

    print("\n--- C code ---")
    print(ex["c"])

    print("\n--- Target ---")
    print(truncate_output(ex["s"]))

    print("\n--- Base ---")
    print(truncate_output(base_outputs[i]))

    print("\n--- SFT ---")
    print(truncate_output(sft_outputs[i]))

    print("\n--- Few-shot ---")
    print(truncate_output(fewshot_outputs[i]))

    print("\n--- GRPO ---")
    print(truncate_output(grpo_outputs[i]))

    print("\n--- PPO ---")
    print(truncate_output(ppo_outputs[i]))


EXAMPLE 1: GCD

--- C code ---
 



#include <stdio.h>
int main() {
    int arr[] = {1,2,3,4,5};
    int n =5;
    int x = 7;
    arr[n] =  x;
    n++;
    for(int i = 0 ; i < n ; i++){
        printf("%d \t" ,arr[i]);
    }
    return 0;
}


--- Target ---
.text
	.section	.rodata
.LC0:
	.string	"%d \t"
	.text
	.globl	main
main:
	pushq	%rbp
	movq	%rsp, %rbp
	subq	$32, %rsp
	movl	$1, -32(%rbp)
	movl	$2, -28(%rbp)
	movl	$3, -24(%rbp)
	movl	$4, -20(%rbp)
	movl	$5, -16(%rbp)
	movl	$5, -8(%rbp)
	movl	$7, -12(%rbp)
	movl	-8(%rbp), %eax
	cltq
	movl	-12(%rbp), %edx
	movl	%edx, -32(%rbp,%rax,4)
	addl	$1, -8(%rbp)
	movl	$0, -4(%rbp)
	jmp	.L2
.L3:
	movl	-4(%rbp), %eax
	cltq
	movl	-32(%rbp,%rax,4), %eax
	movl	%eax, %esi
	leaq	.LC0(%rip), %rax
    ...

--- Base ---


--- SFT ---
.section	.rodata
.LC0:
	.string	"%d \t"
	.text
	.globl	main
main:
	pushq	%rbp
	movq	%rsp, %rbp
	subq	$32, %rsp
	movl	$1, -32(%rbp)
	movl	$2, -28(%rbp)
	movl	$3, -24(%rbp)
	movl	$4, -20(%rbp)
	movl	$5, -16(%rbp)
	movl	$7, -

In [13]:
with open("generation_examples.txt", "w") as f:
    for i, ex in enumerate(examples):
        name = "GCD" if i == 0 else "Factorial"
        f.write(f"\n{'='*80}\n")
        f.write(f"EXAMPLE {i+1}: {name}\n")
        f.write("="*80 + "\n")
        f.write("\n--- C code ---\n")
        f.write(ex["c"] + "\n")
        f.write("\n--- Target ---\n")
        f.write(ex["s"] + "\n")
        f.write("\n--- Base ---\n")
        f.write(base_outputs[i] + "\n")
        f.write("\n--- SFT ---\n")
        f.write(sft_outputs[i] + "\n")
        f.write("\n--- Few-shot ---\n")
        f.write(fewshot_outputs[i] + "\n")
        f.write("\n--- GRPO ---\n")
        f.write(grpo_outputs[i] + "\n")
        f.write("\n--- PPO ---\n")
        f.write(ppo_outputs[i] + "\n")

from google.colab import files
files.download("generation_examples.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>